# 06 — UAV-SEAD: EKF Tutarlılık Feature'ları, Oturum-Split ve Eşik (Faz ML-5)

**Ne değişti:**
- SEAD **179 → 349 uçuş** (199 normal, ExtPos 60, diğer sınıflar 30) — dağılım artık doğala yakın (%57 normal).
- **H13 feature'ları**: `estimator_status` test oranları (PX4'ün *kendi* innovation-tutarlılık kontrolü:
  $r = \frac{\text{innovation}^2}{\text{gate}^2 \cdot S}$, 1'in üstü "ölçüm tahminle tutarsız" demek) +
  `ekf2_innovations` hız/konum innovation büyüklükleri. ALFA'daki `nav_info/errors` hazır-residual
  ilkesinin PX4 karşılığı.
- **Oturum-bazlı split**: aynı günün uçuşları tek split tarafında; anomalili oturumların normalleri
  train/val'den **karantinaya** (test-normal'e) alınır — oturum-içi benzerlik sızıntısı kapandı.

**Ana soru (H13 testi):** ML-4'te adil (ranges) satır-ROC **0.474** idi — UAV-Attack-tasarımlı
feature'lar SEAD'in state-estimation anomalilerini görmüyordu. EKF modülü bunu düzeltiyor mu?

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score
from scipy.stats import genpareto

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
FEAT = ROOT / "data/gold/ml_features"
manifest = json.loads((FEAT / "split_manifest.json").read_text(encoding="utf-8"))

from src.ml.data.scaling import apply_scaler_params
from src.ml.data.splits import session_of

BASE_MODULES = {
    "nav_butunlugu": ["gps_step_m", "log_gps_speed", "gps_accel_mps2", "vertical_rate_calc",
                       "gps_speed_residual", "vertical_rate_residual", "course_change_deg",
                       "gps_step_5s_max", "gps_step_5s_rms", "gps_speed_residual_cusum_pos",
                       "gps_frozen_count"],
    "sinyal_kalitesi": ["jamming_indicator", "noise_per_ms", "hdop", "vdop", "satellites_used",
                         "s_variance_m_s", "eph", "epv", "jamming_1s_mean", "jamming_5s_max",
                         "noise_5s_std", "noise_5s_mean", "hdop_5s_max", "sats_5s_min",
                         "hdop_cusum_pos", "noise_per_ms_cusum_pos"],
    "veri_kalitesi": ["attitude_missing", "battery_missing", "gps_health_missing",
                       "num_missing_groups", "attitude_stale_count", "attitude_stale_s"],
}
EKF_MODULE = {"ekf_tutarlilik": [
    "vel_test_ratio", "pos_test_ratio", "hgt_test_ratio", "mag_test_ratio",
    "tas_test_ratio", "hagl_test_ratio", "beta_test_ratio",
    "ekf_vel_innov_mag", "ekf_pos_innov_mag", "heading_innov",
    "pos_test_ratio_5s_max", "pos_test_ratio_5s_mean", "pos_test_ratio_cusum_pos",
    "vel_test_ratio_5s_max", "vel_test_ratio_5s_mean", "vel_test_ratio_cusum_pos",
    "hgt_test_ratio_5s_max", "hgt_test_ratio_5s_mean", "hgt_test_ratio_cusum_pos",
]}

df_s = pd.read_parquet(FEAT / "uav_sead/uav_sead_ml_features.parquet")
scaler_s = json.loads((ROOT / "artifacts/scalers/uav_sead_robust_scaler.json").read_text())
scaled_s = apply_scaler_params(df_s, scaler_s)
labels_s = manifest["sources"]["uav_sead"]["flight_labels"]
NORMALS = {"normal"}

def anomaly_scores(model, X):
    return -model.score_samples(X)

split0 = manifest["sources"]["uav_sead"]["splits"]["split_00"]
n_norm = sum(1 for v in labels_s.values() if v == "normal")
sess_tr = {session_of(f) for f in split0["train"]}
sess_te = {session_of(f) for f in split0["test"]}
print(f"SEAD: {len(labels_s)} ucus, {n_norm} normal | split_00: train={len(split0['train'])} "
      f"val={len(split0['val'])} test={len(split0['test'])}")
print(f"oturum ayrikligi kontrolu: train-test oturum kesisimi = {len(sess_tr & sess_te)} (0 olmali)")

SEAD: 349 ucus, 199 normal | split_00: train=48 val=21 test=280
oturum ayrikligi kontrolu: train-test oturum kesisimi = 0 (0 olmali)


## 1. H13 ana testi — adil (ranges) satır-ROC: EKF modülü öncesi vs sonrası

Aynı split, aynı IF hiperparametreleri; tek fark feature kümesi. Satır etiketi = satır zamanı
herhangi bir anotasyon aralığının içinde mi (SEAD `ranges`, PX4 µs).

In [2]:
labels_json = json.loads((ROOT / "data/objectstore/bronze/uav_sead/labels.json").read_text(encoding="utf-8"))
flight_ranges = {}
for flight, meta_j in labels_json.items():
    spans = []
    for ann in meta_j.get("ranges", []):
        for sig, ivals in ann:
            spans += [(float(a), float(b)) for a, b in ivals]
    flight_ranges[flight] = spans
n_ranged = sum(1 for v in flight_ranges.values() if v)
print(f"ranges iceren ucus: {n_ranged}/{len(labels_json)}")

sead_silver_t0 = pd.read_parquet(ROOT / "data/silver/uav_sead_silver.parquet",
                                 columns=["source_id", "timestamp"]).groupby("source_id")["timestamp"].min().to_dict()

tr = scaled_s[scaled_s["source_id"].isin(split0["train"])]
va = scaled_s[scaled_s["source_id"].isin(split0["val"])]
te = scaled_s[scaled_s["source_id"].isin(split0["test"])].reset_index(drop=True)

def fused_row_ratio(modules):
    ratio = np.zeros(len(te))
    for mod, cols in modules.items():
        cols = [c for c in cols if c in scaled_s.columns]
        if not cols:
            continue
        m = IsolationForest(n_estimators=300, max_samples=256, random_state=0, n_jobs=-1).fit(tr[cols])
        tau_row = float(np.quantile(anomaly_scores(m, va[cols]), 0.99))
        ratio = np.maximum(ratio, anomaly_scores(m, te[cols]) / tau_row)
    return ratio

# adil satir etiketi
y_fair = np.zeros(len(te), dtype=int)
for sid, g in te.groupby("source_id"):
    spans = flight_ranges.get(sid, [])
    if not spans or sid not in sead_silver_t0:
        continue
    t_raw = sead_silver_t0[sid] + g["t_rel_s"].to_numpy() * 1e6
    hit = np.zeros(len(g), dtype=bool)
    for a, b in spans:
        hit |= (t_raw >= a) & (t_raw <= b)
    y_fair[g.index.to_numpy()] = hit.astype(int)

has_ranges = te["source_id"].map(lambda s: bool(flight_ranges.get(s))).to_numpy()
is_norm = te["source_id"].map(lambda s: labels_s[s] == "normal").to_numpy()
mask_eval = has_ranges | is_norm

r_old = fused_row_ratio(BASE_MODULES)
r_new = fused_row_ratio({**BASE_MODULES, **EKF_MODULE})
r_ekf = fused_row_ratio(EKF_MODULE)
print(f"degerlendirme satiri: {mask_eval.sum()}, aralik-ici anomali satiri: {y_fair[mask_eval].sum()}")
print(f"adil satir-ROC  eski 3 modul        : {roc_auc_score(y_fair[mask_eval], r_old[mask_eval]):.3f}   (ML-4 referansi: 0.474)")
print(f"adil satir-ROC  +EKF modulu (fuzyon): {roc_auc_score(y_fair[mask_eval], r_new[mask_eval]):.3f}")
print(f"adil satir-ROC  yalniz EKF modulu   : {roc_auc_score(y_fair[mask_eval], r_ekf[mask_eval]):.3f}")

ranges iceren ucus: 149/349


degerlendirme satiri: 316956, aralik-ici anomali satiri: 69313
adil satir-ROC  eski 3 modul        : 0.799   (ML-4 referansi: 0.474)
adil satir-ROC  +EKF modulu (fuzyon): 0.781
adil satir-ROC  yalniz EKF modulu   : 0.354


## 2. Uçuş-düzeyi SEAD kendi-tespiti — 5 seed, oturum-split

In [3]:
def run_sead_fusion(modules):
    rows = []
    for split_name, split in manifest["sources"]["uav_sead"]["splits"].items():
        tr = scaled_s[scaled_s["source_id"].isin(split["train"])]
        va = scaled_s[scaled_s["source_id"].isin(split["val"])]
        te = scaled_s[scaled_s["source_id"].isin(split["test"])]
        ratios = {}
        for mod, cols in modules.items():
            cols = [c for c in cols if c in scaled_s.columns]
            if not cols:
                continue
            m = IsolationForest(n_estimators=300, max_samples=256,
                                random_state=split["seed"], n_jobs=-1).fit(tr[cols])
            tau = va.assign(s=anomaly_scores(m, va[cols])).groupby("source_id")["s"].max().max()
            ratios[mod] = te.assign(s=anomaly_scores(m, te[cols])).groupby("source_id")["s"].max() / tau
        R = pd.DataFrame(ratios); R["fusion"] = R.max(axis=1)
        R["label"] = [labels_s[f] for f in R.index]
        R["y"] = (R["label"] != "normal").astype(int)
        rows.append({"split": split_name,
                     "ucus_roc": roc_auc_score(R["y"], R["fusion"]),
                     "tespit@1": float((R["fusion"][R["y"] == 1] > 1).mean()),
                     "FA@1": float((R["fusion"][R["y"] == 0] > 1).mean())})
        if split_name == "split_00":
            per_type = R[R["y"] == 1].groupby("label")["fusion"].apply(lambda x: (x > 1).mean()).round(2)
    return pd.DataFrame(rows).set_index("split"), per_type

res_old, _ = run_sead_fusion(BASE_MODULES)
res_new, per_type = run_sead_fusion({**BASE_MODULES, **EKF_MODULE})
print("SEAD ucus-duzeyi (5 seed ort +- std):")
for adi, r in [("eski 3 modul", res_old), ("+EKF modulu", res_new)]:
    print(f"  {adi:14s} ROC {r['ucus_roc'].mean():.3f} +- {r['ucus_roc'].std():.3f} | "
          f"tespit@1 {r['tespit@1'].mean():.2f} | FA@1 {r['FA@1'].mean():.2f}")
print("\n+EKF tip bazinda tespit@1 (split_00):")
print(per_type.to_string())

SEAD ucus-duzeyi (5 seed ort +- std):
  eski 3 modul   ROC 0.696 +- 0.012 | tespit@1 0.59 | FA@1 0.27
  +EKF modulu    ROC 0.682 +- 0.012 | tespit@1 0.67 | FA@1 0.41

+EKF tip bazinda tespit@1 (split_00):
label
altitude_anomaly             0.27
external_position_anomaly    0.82
global_position_anomaly      0.70
mechanical_fault             0.43


## 3. Eşik: val-max vs POT (artık ~10 val uçuşu ve 199 normal var)

In [ ]:
def pot_threshold_flights(vals, q=0.01, init_quantile=0.5):
    vals = np.asarray(vals)
    u = np.quantile(vals, init_quantile)
    exceed = vals[vals > u] - u
    if len(exceed) < 3:
        return float(np.max(vals))
    xi, _, sigma = genpareto.fit(exceed, floc=0.0)
    n, Nu = len(vals), len(exceed)
    if abs(xi) < 1e-6:
        return float(u + sigma * np.log(Nu / (q * n)))
    return float(u + (sigma / xi) * ((q * n / Nu) ** (-xi) - 1))

rows = []
# Varsayilan model EKF'siz BASE_MODULES'tur; esik karari ayni modelle verilir.
for split_name, split in manifest["sources"]["uav_sead"]["splits"].items():
    tr = scaled_s[scaled_s["source_id"].isin(split["train"])]
    va = scaled_s[scaled_s["source_id"].isin(split["val"])]
    te = scaled_s[scaled_s["source_id"].isin(split["test"])]
    fused_va, fused_te = None, None
    for mod, cols in BASE_MODULES.items():
        cols = [c for c in cols if c in scaled_s.columns]
        m = IsolationForest(n_estimators=300, max_samples=256, random_state=split["seed"], n_jobs=-1).fit(tr[cols])
        va_f = va.assign(s=anomaly_scores(m, va[cols])).groupby("source_id")["s"].max()
        te_f = te.assign(s=anomaly_scores(m, te[cols])).groupby("source_id")["s"].max()
        med = va_f.median() or 1.0
        fused_va = va_f / med if fused_va is None else np.maximum(fused_va, va_f / med)
        fused_te = te_f / med if fused_te is None else np.maximum(fused_te, te_f / med)
    y = np.array([0 if labels_s[f] == "normal" else 1 for f in fused_te.index])
    for esik_adi, tau in [("val-max", float(np.max(fused_va))),
                           ("POT", pot_threshold_flights(fused_va.values))]:
        rows.append({"split": split_name, "esik": esik_adi,
                     "tespit": float((fused_te[y == 1] > tau).mean()),
                     "FA": float((fused_te[y == 0] > tau).mean())})
r = pd.DataFrame(rows)
print(r.groupby("esik")[["tespit", "FA"]].agg(["mean", "std"]).round(3).to_string())

STALE OUTPUT (EKF'li eski kosum) -- hucreyi BASE_MODULES ile yeniden calistirin.
        tespit            FA       
          mean    std   mean    std
esik                               
POT      0.432  0.155  0.225  0.119
val-max  0.400  0.197  0.238  0.145


## 4. Sonuç

- **H13 kararı** bölüm 1-2'deki sayılarla verilir: EKF modülü adil satır-ROC'u ve uçuş tespitini
  anlamlı iyileştirdiyse SEAD artık yalnız kalibrasyon seti değil, kendi anomalilerinin de ölçüldüğü
  üçüncü doğrulama alanıdır.
- Oturum-split ile sonuçlar artık oturum-içi benzerlik iyimserliği taşımıyor (train-test oturum
  kesişimi 0, anomalili oturunların normalleri karantinada).
- Eşik karşılaştırması 199-normalli rejimde POT'un val-max'a karşı konumunu gösterir.
